# 17: Training for real-world images (E8)

E3 and E6 showed that compression, resizing and blur make detectors fail, and E7 showed that recent training data helps. E8 combines both ideas and separates their effects:

| Variant | Training data | Augmentation |
|---|---|---|
| **Full data** | 5 full OpenFake training files (about 20,000 images) | none |
| **Full data + augmentation** | same | random JPEG, resizing and blur during training (as in Wang et al., 2020) |

Both M1 (ResNet-50) and M2 (CLIP + logistic regression) are trained in both variants.

E8 also introduces **full-image analysis**: instead of one centre crop, the image is covered by a grid of 224 x 224 tiles at its original resolution, every tile is scored, and the scores are averaged. Nothing is shrunk, so pixel traces are preserved.

GPU runtime needed; about 33 GB is downloaded. Total time about 90 minutes; processed data and features are saved to Drive, so a restart resumes.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q huggingface_hub

import io
import json
import os
import shutil
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import torch
import torch.nn as nn
import torch.nn.functional as F
from huggingface_hub import hf_hub_download
from PIL import Image, ImageFilter
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss, roc_auc_score
from torch.utils.data import DataLoader, Dataset
from torchvision.models import ResNet50_Weights, resnet50
from transformers import CLIPVisionModelWithProjection

ROOT = Path("/content/drive/MyDrive/minor-project")
RESULTS = ROOT / "results"
DATA = ROOT / "processed_openfake_train_full"
DOWNLOADS = Path("/content/openfake_train")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

## Settings

In [ ]:
TRAIN_GROUPS = [0, 7, 15, 23, 31]   # part 00000 of these training groups (each about 4,000 images)
VAL_FILE = "core/validation-00000-of-00015.parquet"
STORE = 256                          # stored crop size; training uses random 224 crops inside it
SIZE = 224
SEED = 42
EPOCHS = 5
BATCH_SIZE = 64
LEARNING_RATE = 1e-4
NUM_WORKERS = 2
CLIP_NAME = "openai/clip-vit-large-patch14"
MAX_TILES = 9                        # full-image analysis: at most a 3 x 3 grid of tiles

torch.manual_seed(SEED)
np.random.seed(SEED)

## Helper functions

In [ ]:
STD_LUMA = np.array([
    16, 11, 10, 16, 24, 40, 51, 61,
    12, 12, 14, 19, 26, 58, 60, 55,
    14, 13, 16, 24, 40, 57, 69, 56,
    14, 17, 22, 29, 51, 87, 80, 62,
    18, 22, 37, 56, 68, 109, 103, 77,
    24, 35, 55, 64, 81, 104, 113, 92,
    49, 64, 78, 87, 103, 121, 120, 101,
    72, 92, 95, 98, 112, 100, 103, 99,
])


def estimate_quality(img):
    luma = np.array(img.quantization[0])
    best_quality, best_error = 0, np.inf
    for quality in range(1, 101):
        scale = 5000 // quality if quality < 50 else 200 - 2 * quality
        table = np.clip((STD_LUMA * scale + 50) // 100, 1, 255)
        error = np.abs(table - luma).sum()
        if error < best_error:
            best_quality, best_error = quality, error
    return best_quality


def center_crop_aligned(img, size=SIZE):
    width, height = img.size
    left = ((width - size) // 2) // 16 * 16
    top = ((height - size) // 2) // 16 * 16
    return img.crop((left, top, left + size, top + size))


def jpeg_roundtrip(img, quality):
    buffer = io.BytesIO()
    img.save(buffer, format="JPEG", quality=int(quality))
    buffer.seek(0)
    return Image.open(buffer).convert("RGB")


def augment(img, rng):
    """Real-world processing, each step with probability 0.5 (after Wang et al., 2020)."""
    if rng.random() < 0.5:
        img = img.filter(ImageFilter.GaussianBlur(radius=rng.uniform(0.1, 2.0)))
    if rng.random() < 0.5:
        scale = rng.uniform(0.5, 1.0)
        size = img.size[0]
        img = img.resize((round(size * scale),) * 2, Image.BILINEAR).resize((size, size), Image.BILINEAR)
    if rng.random() < 0.5:
        img = jpeg_roundtrip(img, rng.integers(50, 96))
    return img


def tile_positions(width, height, size=SIZE, max_tiles=MAX_TILES):
    """Evenly spaced, 16-pixel-aligned tile corners covering the whole image."""
    per_side = int(np.sqrt(max_tiles))
    xs = np.unique(np.linspace(0, width - size, min(per_side, max(1, width // size))).astype(int) // 16 * 16)
    ys = np.unique(np.linspace(0, height - size, min(per_side, max(1, height // size))).astype(int) // 16 * 16)
    return [(x, y) for y in ys for x in xs]


def tiles_of(img):
    if min(img.size) < SIZE:
        factor = SIZE / min(img.size)
        img = img.resize((max(SIZE, round(img.size[0] * factor)), max(SIZE, round(img.size[1] * factor))), Image.BICUBIC)
    return np.stack([np.asarray(img.crop((x, y, x + SIZE, y + SIZE))) for x, y in tile_positions(*img.size)])

## Download and process the training data (about 50 minutes, only the first time)

Each image is stored as a 256 x 256 centre crop at its original resolution, so that training can take random 224 x 224 crops. Bias control as in notebook 03: non-JPEG images get one JPEG compression at a quality drawn from the real training images.

In [ ]:
def process_file(path, split_name):
    pf = pq.ParquetFile(path)
    records, crops = [], []
    for g in range(pf.num_row_groups):
        t = pf.read_row_group(g, columns=["image", "label", "model"])
        for item, label, model in zip(t["image"].to_pylist(), t["label"].to_pylist(), t["model"].to_pylist()):
            img = Image.open(io.BytesIO(item["bytes"]))
            if img.mode in ("L", "LA", "1", "I", "I;16") or min(img.size) < STORE:
                continue
            records.append({"split": split_name, "source": model, "label": label, "format": img.format,
                            "jpeg_quality": estimate_quality(img) if img.format == "JPEG" else np.nan})
            crops.append(np.asarray(center_crop_aligned(img.convert("RGB"), STORE)))
    return pd.DataFrame(records), np.stack(crops)


if not (DATA / "meta.csv").exists():
    picks = [("train", f"core/train-{g:05d}-of-00032-00000.parquet") for g in TRAIN_GROUPS] + [("val", VAL_FILE)]
    metas, arrays = [], []
    DOWNLOADS.mkdir(exist_ok=True)
    for split_name, name in picks:
        start = time.time()
        path = hf_hub_download("ComplexDataLab/OpenFake", name, repo_type="dataset", local_dir=DOWNLOADS)
        m, a = process_file(path, split_name)
        metas.append(m)
        arrays.append(a)
        os.remove(path)
        print(f"{name}: {len(m)} usable images ({time.time() - start:.0f}s)")
    meta = pd.concat(metas, ignore_index=True)
    images = np.concatenate(arrays)
    del arrays

    keep = []
    for split_name, group in meta.groupby("split"):
        n = min((group["label"] == "real").sum(), (group["label"] == "fake").sum())
        keep += group[group["label"] == "real"].sample(n=n, random_state=SEED).index.tolist()
        keep += group[group["label"] == "fake"].sample(n=n, random_state=SEED).index.tolist()
    keep = sorted(keep)
    meta, images = meta.loc[keep].reset_index(drop=True), images[keep]

    real_q = meta.loc[(meta["label"] == "real") & (meta["split"] == "train"), "jpeg_quality"].dropna().to_numpy()
    rng = np.random.default_rng(SEED)
    meta["compressed_by_us"] = meta["format"] != "JPEG"
    for i in meta.index[meta["compressed_by_us"]]:
        q = int(rng.choice(real_q))
        images[i] = np.asarray(jpeg_roundtrip(Image.fromarray(images[i]), q))
        meta.loc[i, "jpeg_quality"] = q

    DATA.mkdir(exist_ok=True)
    np.save(DATA / "images.npy", images)
    meta.to_csv(DATA / "meta.csv", index=False)
    del images

meta = pd.read_csv(DATA / "meta.csv")
images = np.load(DATA / "images.npy", mmap_mode="r")
meta["y"] = (meta["label"] == "fake").astype(int)
print(meta.groupby(["split", "label"]).size().unstack())
display(meta.groupby(["split", "label"])["jpeg_quality"].describe()[["mean", "50%"]].round(1))
print(f"Generators in training data: {meta[meta['y'] == 1]['source'].nunique()}")

## Training and test data loaders

In [ ]:
class TrainSet(Dataset):
    """Random 224 crop from the stored 256 crop, random flip, optional real-world augmentation."""
    def __init__(self, rows, augmented):
        self.rows, self.augmented = rows, augmented

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        rng = np.random.default_rng()
        row = self.rows[i]
        x, y = rng.integers(0, STORE - SIZE + 1, size=2)
        img = Image.fromarray(np.array(images[row])).crop((x, y, x + SIZE, y + SIZE))
        if rng.random() < 0.5:
            img = img.transpose(Image.FLIP_LEFT_RIGHT)
        if self.augmented:
            img = augment(img, rng)
        return torch.from_numpy(np.asarray(img).copy()), meta["y"].iat[row]


def center_224(rows):
    off = (STORE - SIZE) // 2
    return np.stack([np.array(images[r])[off:off + SIZE, off:off + SIZE] for r in rows])


train_rows = meta.index[meta["split"] == "train"].to_numpy()
val_rows = meta.index[meta["split"] == "val"].to_numpy()
y_all = meta["y"].to_numpy()
val_images = center_224(val_rows)

of_images = np.load(ROOT / "processed_openfake" / "images.npy")
of_meta = pd.read_csv(RESULTS / "e7_openfake_predictions.csv")
gi_images = np.load(ROOT / "processed" / "test_images.npy", mmap_mode="r")
gi_y = (pd.read_csv(ROOT / "processed" / "test_meta.csv")["label"] == "fake").astype(int).to_numpy()
wa = pd.read_csv(RESULTS / "e6_whatsapp_predictions.csv")
received_dir = ROOT / "whatsapp" / "received"
wa_files = {"before": [ROOT / "whatsapp" / "original" / f for f in wa["file"]],
            "after": [received_dir / f for f in wa["received_file"]]}
print(f"train {len(train_rows)}, val {len(val_rows)}, OpenFake test {len(of_meta)}, GenImage test {len(gi_y)}, WhatsApp pairs {len(wa)}")

## M1 (ResNet-50): train both variants

In [ ]:
MEAN = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)
use_amp = device.type == "cuda"


def to_input(batch):
    return (batch.to(device, non_blocking=True).permute(0, 3, 1, 2).float() / 255 - MEAN) / STD


@torch.no_grad()
def predict_resnet(net, arr):
    net.eval()
    out = []
    for s in range(0, len(arr), BATCH_SIZE * 2):
        with torch.autocast(device.type, enabled=use_amp):
            out.append(torch.sigmoid(net(to_input(torch.from_numpy(np.array(arr[s:s + BATCH_SIZE * 2])))).squeeze(1).float()).cpu())
    return torch.cat(out).numpy()


def train_resnet(name, augmented):
    ckpt = ROOT / "checkpoints" / f"{name}.pt"
    net = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
    net.fc = nn.Linear(net.fc.in_features, 1)
    net = net.to(device)
    if ckpt.exists():
        net.load_state_dict(torch.load(ckpt, map_location=device))
        print(f"{name}: loaded saved checkpoint")
        return net
    loader = DataLoader(TrainSet(train_rows, augmented), batch_size=BATCH_SIZE, shuffle=True,
                        num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=NUM_WORKERS > 0)
    opt = torch.optim.AdamW(net.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
    scaler = torch.amp.GradScaler(device.type, enabled=use_amp)
    best = float("inf")
    for epoch in range(1, EPOCHS + 1):
        net.train()
        start, running = time.time(), 0.0
        for batch, labels in loader:
            with torch.autocast(device.type, enabled=use_amp):
                loss = F.binary_cross_entropy_with_logits(net(to_input(batch)).squeeze(1), labels.float().to(device))
            opt.zero_grad()
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            running += loss.item() * len(labels)
        val_probs = predict_resnet(net, val_images)
        val_loss = float(log_loss(y_all[val_rows], val_probs, labels=[0, 1]))
        print(f"{name} epoch {epoch}: train loss {running / len(train_rows):.4f} | val loss {val_loss:.4f} | "
              f"val AUC {roc_auc_score(y_all[val_rows], val_probs):.4f} | {time.time() - start:.0f}s")
        if val_loss < best:
            best = val_loss
            torch.save(net.state_dict(), ckpt)
            print("  saved as best so far")
    net.load_state_dict(torch.load(ckpt, map_location=device))
    return net


m1 = {"full data": train_resnet("e8_m1_full", augmented=False),
      "full data + augmentation": train_resnet("e8_m1_full_aug", augmented=True)}

## M2 (CLIP + logistic regression): both variants

For the augmented variant, CLIP features are computed for the clean training crops plus two augmented copies of each, and the classifier is trained on all of them.

In [ ]:
dtype = torch.float16 if device.type == "cuda" else torch.float32
clip = CLIPVisionModelWithProjection.from_pretrained(CLIP_NAME, torch_dtype=dtype).to(device).eval()
CLIP_SIZE = clip.config.image_size
CLIP_MEAN = torch.tensor([0.48145466, 0.4578275, 0.40821073], device=device).view(1, 3, 1, 1)
CLIP_STD = torch.tensor([0.26862954, 0.26130258, 0.27577711], device=device).view(1, 3, 1, 1)


@torch.no_grad()
def clip_features(arr):
    out = []
    for s in range(0, len(arr), 128):
        x = torch.from_numpy(np.array(arr[s:s + 128])).to(device).permute(0, 3, 1, 2).float() / 255
        if x.shape[-1] != CLIP_SIZE:
            x = F.interpolate(x, size=CLIP_SIZE, mode="bicubic", align_corners=False)
        emb = clip(pixel_values=((x - CLIP_MEAN) / CLIP_STD).to(dtype)).image_embeds.float()
        out.append(F.normalize(emb, dim=1).cpu().numpy())
    return np.concatenate(out)


def cached(name, make):
    path = DATA / f"{name}.npy"
    if not path.exists():
        np.save(path, make())
    return np.load(path)


def augmented_copy(seed):
    rng = np.random.default_rng(seed)
    out = np.empty((len(train_rows), SIZE, SIZE, 3), dtype=np.uint8)
    off = (STORE - SIZE) // 2
    for k, r in enumerate(train_rows):
        img = Image.fromarray(np.array(images[r])[off:off + SIZE, off:off + SIZE])
        out[k] = np.asarray(augment(img, rng))
    return out


f_train = cached("clip_train", lambda: clip_features(center_224(train_rows)))
f_val = cached("clip_val", lambda: clip_features(val_images))
f_aug = [cached(f"clip_train_aug{s}", lambda s=s: clip_features(augmented_copy(s))) for s in (1, 2)]
y_train, y_val = y_all[train_rows], y_all[val_rows]


def fit_lr(X, y):
    search = []
    for C in [0.1, 1, 10, 100]:
        c = LogisticRegression(C=C, max_iter=5000).fit(X, y)
        search.append((float(log_loss(y_val, c.predict_proba(f_val)[:, 1])), C))
    C = min(search)[1]
    print("  validation loss per C:", [(c, round(l, 4)) for l, c in search], "-> C =", C)
    return LogisticRegression(C=C, max_iter=5000).fit(X, y)


print("M2 full data:")
m2_plain = fit_lr(f_train, y_train)
print("M2 full data + augmentation:")
m2_aug = fit_lr(np.concatenate([f_train] + f_aug), np.concatenate([y_train] * 3))
m2 = {"full data": m2_plain, "full data + augmentation": m2_aug}

## Evaluation: centre crop (comparable with E6 and E7)

In [ ]:
of_feats = clip_features(of_images)
gi_feats = cached("clip_genimage_test", lambda: clip_features(gi_images))
of_y = of_meta["y"].to_numpy()

rows = []
for test, y, arr, feats in [("openfake", of_y, of_images, of_feats), ("genimage", gi_y, gi_images, gi_feats)]:
    for variant in m1:
        for model, p in [("M1", predict_resnet(m1[variant], arr)), ("M2", m2[variant].predict_proba(feats)[:, 1])]:
            rows.append({"test": test, "model": model, "variant": variant, "auc": float(roc_auc_score(y, p)),
                         "real_called_real": float((p[y == 0] < 0.5).mean()), "fake_detected": float((p[y == 1] >= 0.5).mean())})
            if test == "openfake":
                of_meta[f"{model} {variant}"] = p
e7 = pd.read_csv(RESULTS / "e7_summary.csv")
e7 = e7[e7["trained_on"].str.startswith("OpenFake") & e7["test"].isin(["openfake", "genimage"])]
for r in e7.itertuples():
    rows.append({"test": r.test, "model": r.model, "variant": "E7 (about 4,000 images)", "auc": r.auc,
                 "real_called_real": r.real_called_real, "fake_detected": r.fake_detected})
center = pd.DataFrame(rows)
center.to_csv(RESULTS / "e8_center_summary.csv", index=False)
of_meta.to_csv(RESULTS / "e8_openfake_predictions.csv", index=False)
display(center.pivot_table(index=["test", "model"], columns="variant", values="auc").round(3))

cols = ["M1 new", "M2 new"] + [c for c in of_meta.columns if c.startswith(("M1 full", "M2 full"))]
correct = pd.DataFrame((of_meta[cols].to_numpy() >= 0.5) == of_y[:, None], columns=cols)
per_source = correct.groupby(of_meta["source"]).mean().rename(columns={"M1 new": "M1 E7", "M2 new": "M2 E7"})
per_source.to_csv(RESULTS / "e8_openfake_per_source.csv")
display(per_source.round(3))

## Evaluation: full image vs centre crop, on the WhatsApp images

Each image is covered by up to 3 x 3 tiles of 224 x 224 pixels at original resolution; the tile scores are averaged.

In [ ]:
def score_image(path, variant, model, full):
    img = Image.open(path).convert("RGB")
    arr = tiles_of(img) if full else np.asarray(center_crop_aligned(img))[None]
    if model == "M1":
        return float(predict_resnet(m1[variant], arr).mean())
    return float(m2[variant].predict_proba(clip_features(arr))[:, 1].mean())


wa_y = wa["y"].to_numpy()
rows = []
for variant in m1:
    for model in ["M1", "M2"]:
        for when in ["before", "after"]:
            for full in [False, True]:
                p = np.array([score_image(f, variant, model, full) for f in wa_files[when]])
                rows.append({"variant": variant, "model": model, "when": when, "view": "full image" if full else "centre crop",
                             "auc": float(roc_auc_score(wa_y, p)), "real_called_real": float((p[wa_y == 0] < 0.5).mean()),
                             "fake_detected": float((p[wa_y == 1] >= 0.5).mean())})
whatsapp = pd.DataFrame(rows)
whatsapp.to_csv(RESULTS / "e8_whatsapp_summary.csv", index=False)
display(whatsapp.pivot_table(index=["variant", "model"], columns=["when", "view"], values="auc").round(3))

## Chart

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.6), sharey=True)
panels = [("OpenFake test (centre crop)", center[center["test"] == "openfake"], "variant"),
          ("WhatsApp, after sharing (full image)", whatsapp[(whatsapp["when"] == "after") & (whatsapp["view"] == "full image")], "variant"),
          ("GenImage test (centre crop)", center[center["test"] == "genimage"], "variant")]
variants = [("E7 (about 4,000 images)", "#a9b4c2"), ("full data", "#7fa7dd"), ("full data + augmentation", "#2a78d6")]
x, w = np.arange(2), 0.27
for ax, (title, part, _) in zip(axes, panels):
    for k, (variant, color) in enumerate(variants):
        sub = part[part["variant"] == variant].set_index("model")
        if sub.empty:
            continue
        values = sub.loc[["M1", "M2"], "auc"] * 100
        bars = ax.bar(x + (k - 1) * w, values, w - 0.03, color=color, label=variant)
        ax.bar_label(bars, fmt="%.0f", fontsize=8)
    ax.set_xticks(x, ["M1 ResNet-50", "M2 CLIP"])
    ax.set_title(title, fontsize=10)
    ax.axhline(50, color="#999999", linewidth=1, linestyle="--")
    ax.grid(axis="y", color="#e5e5e5")
    ax.set_axisbelow(True)
    ax.spines[["top", "right"]].set_visible(False)
axes[0].set_ylabel("ROC-AUC x 100")
axes[0].set_ylim(0, 105)
h, l = axes[0].get_legend_handles_labels()
fig.legend(h, l, loc="upper center", ncol=3, frameon=False)
plt.tight_layout(rect=[0, 0, 1, 0.9])
plt.savefig(RESULTS / "e8_robust.png", dpi=120)
plt.show()